In [20]:
import socket
import time

probes = {
    "BaseProbe1": b"\x00\x0e\x38" + b"\x00" * 13,
    "BaseProbe2": b"\x00\x0e\x38" + b"\x00" * 12,  # missing 1 byte
    "Generic": b"\x0d\x0a\x0d\x0a",
    "OneZero": b"\x00",
    "TwoZero": b"\x00\x00",
    "EPMD": b"\x00\x01\x6e",
    "SSH": b"SSH-2.0-OpenSSH_8.1\r\n",
    "HTTP-GET": b"GET / HTTP/1.0\r\n\r\n",
    "TLS": b"\x16\x03\x01\x00\x6f" + b"\x01" * 111,
    "2K-Random": bytes([0x41] * 2000),
}

def send_probe(ip, port, payload, timeout=3):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        start = time.time()
        s.connect((ip, port))
        s.sendall(payload)
        try:
            response = s.recv(1024)
            end = time.time()
            return {
                "response_len": len(response),
                "response_hex": response.hex()[:100],
                "time_elapsed": round(end - start, 4)
            }
        except socket.timeout:
            return {"response_len": 0, "response_hex": '', "time_elapsed": round(time.time() - start, 4)}
        finally:
            s.close()
    except Exception as e:
        return {"response_len": -1, "error": str(e), "time_elapsed": -1}

def classify(results):
    b1 = results.get("BaseProbe1", {})
    b2 = results.get("BaseProbe2", {})
    resp_b1 = b1.get("response_len", 0)
    resp_b2 = b2.get("response_len", 0)
    t_diff = b2.get("time_elapsed", 0) - b1.get("time_elapsed", 0)

    if resp_b1 == 0 and resp_b2 == 0 and t_diff > 1.0:
        return "✅ Likely OpenVPN with tls-auth / tls-crypt (probe-resistant)"
    elif resp_b1 == 28 and resp_b2 == 28:
        return "🟡 Likely Vanilla OpenVPN (no tls-auth)"
    elif any(results.get(p, {}).get("response_len", 0) > 0 for p in ["HTTP-GET", "SSH", "TLS", "Generic"]):
        return "❌ Likely not OpenVPN (web or SSH service)"
    elif resp_b1 == 0 and resp_b2 == 0:
        return "⚠️ No response — could be obfuscated OpenVPN or unrelated"
    else:
        return "❓ Unknown behavior — inconclusive"

def run_probe_and_classify(ip, port=1194):
    print(f"🔍 Probing {ip}:{port}")
    results = {}
    for probe_name, payload in probes.items():
        print(f"→ Sending {probe_name}...")
        result = send_probe(ip, port, payload)
        results[probe_name] = result
        print(f"  Response: len={result['response_len']}, time={result['time_elapsed']}s")

    print("\n📊 Classification Result:")
    verdict = classify(results)
    print(f"🔎 Verdict: {verdict}")
    return verdict

# Example usage
if __name__ == "__main__":
    target_ip = input("Enter target IP: ").strip()
    target_port = 1194
    run_probe_and_classify(target_ip, target_port)


Enter target IP:  62.91.112.155


🔍 Probing 62.91.112.155:1194
→ Sending BaseProbe1...
  Response: len=-1, time=-1s
→ Sending BaseProbe2...
  Response: len=-1, time=-1s
→ Sending Generic...
  Response: len=-1, time=-1s
→ Sending OneZero...
  Response: len=-1, time=-1s
→ Sending TwoZero...
  Response: len=-1, time=-1s
→ Sending EPMD...
  Response: len=-1, time=-1s
→ Sending SSH...
  Response: len=-1, time=-1s
→ Sending HTTP-GET...
  Response: len=-1, time=-1s
→ Sending TLS...
  Response: len=-1, time=-1s
→ Sending 2K-Random...
  Response: len=-1, time=-1s

📊 Classification Result:
🔎 Verdict: ❓ Unknown behavior — inconclusive
